In [1]:
library(fixest)
library(modelsummary)
library(gt)
library(marginaleffects)
library(ggeffects)
library(ggplot2)
library(dplyr)
library(sjPlot)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
df_path='../../data/dropbox/LaborDivision/data/deriveddata/intl_noalpha/regression_table.csv'
data=read.csv(df_path)

In [3]:
head(data)

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,ST,funded
,<int>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,29768070,65683243,JPN,F,2008,51,F,2008,5,Advanced,invalid
2,31240446,65683243,JPN,F,2008,51,F,2008,5,Advanced,invalid
3,29768070,79129017,JPN,M,2008,51,M,2000,98,Advanced,invalid
4,31240446,79129017,JPN,M,2008,51,M,2000,98,Advanced,invalid
5,39595623,79129017,JPN,F,2013,18,M,2000,98,Advanced,1
6,40710917,79129017,JPN,F,2013,51,M,2000,98,Advanced,1


In [4]:
data$npub=log(data$npub)
data$funded <- factor(data$funded, levels = c(0, 1), labels = c("NotFund", "Funded"))
data$fdep=ifelse(data$role == 'F', 1, 0)
data$ldep=ifelse(data$role == 'L', 1, 0)
data$mdep=ifelse(data$role == 'M', 1, 0)

In [5]:
head(data)

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,ST,funded,fdep,ldep,mdep
,<int>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,29768070,65683243,JPN,F,2008,51,F,2008,1.609438,Advanced,NA,1,0,0
2,31240446,65683243,JPN,F,2008,51,F,2008,1.609438,Advanced,NA,1,0,0
3,29768070,79129017,JPN,M,2008,51,M,2000,4.584967,Advanced,NA,0,0,1
4,31240446,79129017,JPN,M,2008,51,M,2000,4.584967,Advanced,NA,0,0,1
5,39595623,79129017,JPN,F,2013,18,M,2000,4.584967,Advanced,Funded,1,0,0
6,40710917,79129017,JPN,F,2013,51,M,2000,4.584967,Advanced,Funded,1,0,0


In [6]:
unique_ids=unique(data$ost_bk)
sample_id=sample(unique_ids,1000000)
sample_data=data[data$ost_bk %in% sample_id,]

In [7]:
length(unique(data$ost_bk))

[1] 3195989

In [8]:
fund_data = data[!is.na(data$funded),]
unique_ids=unique(fund_data$ost_bk)
sample_id=sample(unique_ids,10000)
sample_fund=fund_data[fund_data$ost_bk %in% sample_id,]

In [9]:
head(sample_data)

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,ST,funded,fdep,ldep,mdep
,<int>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
2,31240446,65683243,JPN,F,2008,51,F,2008,1.609438,Advanced,NA,1,0,0
4,31240446,79129017,JPN,M,2008,51,M,2000,4.584967,Advanced,NA,0,0,1
5,39595623,79129017,JPN,F,2013,18,M,2000,4.584967,Advanced,Funded,1,0,0
6,40710917,79129017,JPN,F,2013,51,M,2000,4.584967,Advanced,Funded,1,0,0
8,44297811,79129017,JPN,F,2015,51,M,2000,4.584967,Advanced,NA,1,0,0
12,34414334,9735544,JPN,M,2010,41,M,1975,4.990433,Advanced,Funded,0,0,1


In [10]:
head(sample_fund)

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,ST,funded,fdep,ldep,mdep
,<int>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
429,101014951,91035604,JPN,M,2010,28,M,1998,5.676754,Advanced,Funded,0,0,1
468,31573005,18018376,JPN,M,2009,31,M,1996,5.605802,Advanced,NotFund,0,0,1
569,31573005,33865810,JPN,M,2009,31,M,1990,5.068904,Advanced,NotFund,0,0,1
735,36450323,76099244,JPN,M,2011,68,F,2005,3.401197,Advanced,Funded,0,0,1
1467,97960271,48447827,JPN,M,2011,48,M,1982,7.043160,Advanced,NotFund,0,0,1
1683,56145650,50668177,JPN,M,2017,82,M,1993,7.576097,Advanced,Funded,0,0,1


## model with fixed effect with fixest package

In [11]:
model1 <- feglm(fdep~gender+npub+ST|ost_bk,data = sample_data,family = binomial())

In [12]:
summary(model1)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 6,552,648 
Fixed-effects: ost_bk: 1,000,000
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error   t value  Pr(>|t|)    
genderM      -0.267285   0.003433  -77.8535 < 2.2e-16 ***
genderU      -0.294919   0.004870  -60.5521 < 2.2e-16 ***
npub         -0.225588   0.000813 -277.5444 < 2.2e-16 ***
STDeveloping  0.536629   0.008504   63.1063 < 2.2e-16 ***
STLagging     0.220423   0.007937   27.7719 < 2.2e-16 ***
STProficient  0.617106   0.004250  145.1846 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -2,556,991.2   Adj. Pseudo R2: -0.270648
           BIC: 20,809,456.4     Squared Cor.:  0.081454

In [13]:
model2 <- feglm(fdep~gender+npub+ST+funded|ost_bk,data = sample_fund,family = binomial())
summary(model2)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 65,409 
Fixed-effects: ost_bk: 10,000
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error   t value   Pr(>|t|)    
genderM      -0.199895   0.035009  -5.70989 1.1305e-08 ***
genderU      -0.235524   0.048803  -4.82606 1.3926e-06 ***
npub         -0.233775   0.007993 -29.24923  < 2.2e-16 ***
STDeveloping  1.020659   0.110170   9.26444  < 2.2e-16 ***
STLagging     0.500699   0.093653   5.34630 8.9772e-08 ***
STProficient  0.813461   0.048412  16.80304  < 2.2e-16 ***
fundedFunded  1.004091   0.040279  24.92857  < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -25,176.0   Adj. Pseudo R2: -0.257663
           BIC: 161,313.7     Squared Cor.:  0.093384

In [14]:
model3 <- feglm(fdep~gender+npub+ST+funded+ST*funded|ost_bk,data = sample_fund,family = binomial())
summary(model3)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 65,409 
Fixed-effects: ost_bk: 10,000
Standard-errors: Clustered (ost_bk) 
                           Estimate Std. Error    t value   Pr(>|t|)    
genderM                   -0.201220   0.035017  -5.746391 9.1168e-09 ***
genderU                   -0.236484   0.048836  -4.842430 1.2826e-06 ***
npub                      -0.233588   0.008004 -29.184528  < 2.2e-16 ***
STDeveloping               0.877978   0.145067   6.052214 1.4287e-09 ***
STLagging                  0.476815   0.124329   3.835097 1.2551e-04 ***
STProficient               0.505303   0.092494   5.463068 4.6797e-08 ***
fundedFunded               0.895266   0.052245  17.135836  < 2.2e-16 ***
STDeveloping:fundedFunded  0.259277   0.222124   1.167262 2.4310e-01    
STLagging:fundedFunded    -0.072803   0.178101  -0.408776 6.8270e-01    
STProficient:fundedFunded  0.451617   0.113422   3.981734 6.8414e-05 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 

In [15]:
model4 <- feglm(mdep~gender+npub+ST|ost_bk,data = sample_data,family = binomial())
model5 <- feglm(mdep~gender+npub+ST+funded|ost_bk,data = sample_fund,family = binomial())
model6 <- feglm(mdep~gender+npub+ST+funded+ST*funded|ost_bk,data = sample_fund,family = binomial())

NOTE: 40,922 fixed-effects (81,844 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 340 fixed-effects (680 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 340 fixed-effects (680 observations) removed because of only 0 (or only 1) outcomes.



In [16]:
model7 <- feglm(ldep~gender+npub+ST|ost_bk,data = sample_data,family = binomial())
summary(model7)

GLM estimation, family = binomial, Dep. Var.: ldep
Observations: 6,552,648 
Fixed-effects: ost_bk: 1,000,000
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error  t value  Pr(>|t|)    
genderM       0.224828   0.003926  57.2685 < 2.2e-16 ***
genderU       0.196873   0.005288  37.2288 < 2.2e-16 ***
npub          0.496240   0.001148 432.1322 < 2.2e-16 ***
STDeveloping -0.601283   0.010590 -56.7793 < 2.2e-16 ***
STLagging    -0.411600   0.009647 -42.6653 < 2.2e-16 ***
STProficient -0.249724   0.004979 -50.1581 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -2,421,271.0   Adj. Pseudo R2: -0.222165
           BIC: 20,538,016.0     Squared Cor.:  0.121987

In [17]:
model8 <- feglm(ldep~gender+npub+ST+funded|ost_bk,data = sample_fund,family = binomial())
summary(model8)

GLM estimation, family = binomial, Dep. Var.: ldep
Observations: 65,409 
Fixed-effects: ost_bk: 10,000
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error  t value   Pr(>|t|)    
genderM       0.216282   0.039569  5.46595 4.6043e-08 ***
genderU       0.163731   0.052506  3.11830 1.8190e-03 ** 
npub          0.517745   0.011624 44.53971  < 2.2e-16 ***
STDeveloping -0.459775   0.120601 -3.81235 1.3765e-04 ***
STLagging    -0.545624   0.106662 -5.11543 3.1302e-07 ***
STProficient -0.330431   0.050244 -6.57654 4.8152e-11 ***
fundedFunded  0.219213   0.039548  5.54297 2.9738e-08 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -24,137.5   Adj. Pseudo R2: -0.220539
           BIC: 159,236.7     Squared Cor.:  0.125058

In [18]:
model9 <- feglm(ldep~gender+npub+ST+funded+ST*funded|ost_bk,data = sample_fund,family = binomial())
summary(model9)

GLM estimation, family = binomial, Dep. Var.: ldep
Observations: 65,409 
Fixed-effects: ost_bk: 10,000
Standard-errors: Clustered (ost_bk) 
                           Estimate Std. Error   t value   Pr(>|t|)    
genderM                    0.215903   0.039593  5.452989 4.9530e-08 ***
genderU                    0.162585   0.052540  3.094494 1.9715e-03 ** 
npub                       0.518178   0.011634 44.538234  < 2.2e-16 ***
STDeveloping              -0.132163   0.162749 -0.812065 4.1675e-01    
STLagging                 -0.363174   0.135330 -2.683607 7.2833e-03 ** 
STProficient              -0.208173   0.098611 -2.111055 3.4768e-02 *  
fundedFunded               0.311989   0.050067  6.231380 4.6234e-10 ***
STDeveloping:fundedFunded -0.643157   0.228012 -2.820720 4.7916e-03 ** 
STLagging:fundedFunded    -0.320985   0.198080 -1.620478 1.0513e-01    
STProficient:fundedFunded -0.190739   0.117772 -1.619563 1.0533e-01    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
L

In [22]:
models = list("(1)"=model1,"(2)"=model2,"(1)"=model4,"(2)"=model5,"(1)"=model7,"(2)"=model8)
table = modelsummary(models, output='gt', stars=TRUE,
            coef_map=c('genderM'='Male','npub'='Log no.pub',
                          'STProficient'='Proficient','STDeveloping'='Developing','STLagging'='Lagging',
                          'fundedFunded'='Funded'),
            statistic=NULL,
            coef_omit=c('genderU'),
            gof_map = c("nobs", "r.squared","adj.r.squared"))
table |> 
tab_spanner(label='First Author',columns=c(2:3)) |> 
tab_spanner(label='Middle Author',columns=c(4:5))|>
tab_spanner(label='Last Author',columns=c(6:7))|>
gtsave(file='regression_poster.html')

In [34]:
models = list("(1)"=model1,"(2)"=model2,"(3)"=model3,"(1)"=model4,"(2)"=model5,"(3)"=model6,"(1)"=model7,"(2)"=model8,"(3)"=model9)
table = modelsummary(models, output='gt', stars=TRUE,
            coef_map=c('genderM'='Male','genderU'='Gender(Unknown)','npub'='Log no.pub',
                          'STProficient'='Proficient','STDeveloping'='Developing','STLagging'='Lagging',
                          'fundedFunded'='Funded',
                          'STProficient:fundedFunded'='Proficient × Funded',
                          'STDeveloping:fundedFunded'='Developing × Funded',
                          'STLagging:fundedFunded'='Lagging × Funded'),
            gof_map = c("nobs", "r.squared","adj.r.squared"))
table |> 
tab_spanner(label='First Author',columns=c(2:4)) |> 
tab_spanner(label='Middle Author',columns=c(5:7))|>
tab_spanner(label='Last Author',columns=c(8:10))|>
gtsave(file='regression.html')

In [17]:
summary(model3)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 66,463 
Fixed-effects: ost_bk: 10,000
Standard-errors: Clustered (ost_bk) 
                           Estimate Std. Error    t value   Pr(>|t|)    
genderM                   -0.300455   0.034567  -8.691919  < 2.2e-16 ***
genderU                   -0.306774   0.048983  -6.262897 3.7789e-10 ***
npub                      -0.241064   0.008009 -30.098998  < 2.2e-16 ***
STDeveloping               0.471020   0.138894   3.391218 6.9583e-04 ***
STLagging                  0.289200   0.125951   2.296134 2.1668e-02 *  
STProficient               0.408935   0.091637   4.462558 8.0987e-06 ***
fundedFunded               0.822937   0.052201  15.764918  < 2.2e-16 ***
STDeveloping:fundedFunded  0.612312   0.219049   2.795320 5.1848e-03 ** 
STLagging:fundedFunded     0.021494   0.176714   0.121634 9.0319e-01    
STProficient:fundedFunded  0.477342   0.111358   4.286569 1.8145e-05 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 

In [23]:
options(repr.plot.width=12, repr.plot.height=4)

In [24]:
color_scheme_path='../../data/dropbox/LaborDivision/data/additionaldata/cntry_classification/color_scheme.csv'
color_scheme=read.csv(color_scheme_path)

In [25]:
color_scheme = color_scheme[color_scheme$type == "st", ]

In [26]:
color_palette <- setNames(color_scheme$color, color_scheme$class)
color_palette

Advanced Proficient Developing    Lagging 
 "#D96C94"  "#F88A4D"  "#5DC97C"  "#23A5D9"

In [41]:
plot = plot_model(model3, type='pred', 
           terms=c("funded","ST [Advanced, Proficient,Developing,Lagging]","gender [F,M,U]"),
          title="",
          axis.title=c("",'Probability of being first author'),
          legend.title='Scientific Capacity',
          term_labels=c('F'='Female'))+
set_theme(axis.textsize=1.1)+
theme(axis.text.x = element_text(angle=0),
      panel.spacing=unit(3.5,"lines"),
      panel.background = element_blank(),
      panel.grid.major=element_line(linewidth = 0.0,color = "white"),
      panel.grid.minor=element_line(linewidth = 0.0,color = "white"),
      strip.background = element_blank()
      )+
scale_color_manual(values = color_palette)
ggsave("MEF_First_plot.pdf", plot, width = 7, height = 3)

Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.


In [28]:
plot = plot_model(model6, type='pred', 
           terms=c("funded","ST [Advanced, Proficient,Developing,Lagging]","gender [F,M,U]"),
          title="",
          axis.title=c("",'Probability of being middle author'),
          legend.title='Scientific Capacity',
          term_labels=c('F'='Female'))+
set_theme(axis.textsize=1.1)+
theme(axis.text.x = element_text(angle=0),
      panel.spacing=unit(2,"lines"),
      panel.background = element_blank(),
      panel.grid.major=element_line(linewidth = 0.1,color = "grey"),
      panel.grid.minor=element_line(linewidth = 0.1,color = "grey"),
      strip.background = element_blank()
      )+
scale_color_manual(values = color_palette)
ggsave("MEF_Middle_plot.pdf", plot, width = 12, height = 4)

Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.


In [43]:
plot = plot_model(model9, type='pred', 
           terms=c("funded","ST [Advanced, Proficient,Developing,Lagging]","gender [F,M,U]"),
          title="",
          axis.title=c("",'Probability of being last author'),
          legend.title='Scientific Capacity',
          term_labels=c('F'='Female'))+
set_theme(axis.textsize=1.1)+
theme(axis.text.x = element_text(angle=0),
      panel.spacing=unit(3.5,"lines"),
      panel.background = element_blank(),
      panel.grid.major=element_line(linewidth = 0.1,color = "white"),
      panel.grid.minor=element_line(linewidth = 0.1,color = "white"),
      strip.background = element_blank()
      )+
scale_color_manual(values = color_palette)
ggsave("MEF_Last_plot.pdf", plot, width = 7, height = 3)

Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.


In [49]:
plot = plot_model(model9, type='pred', 
           terms=c("funded","ST [Advanced, Proficient,Developing,Lagging]"),
          title="",
          axis.title=c("",''),
          legend.title='Scientific Capacity')+
set_theme(axis.textsize=1.1)+
theme(axis.text.x = element_text(angle=0),
      panel.spacing=unit(3.5,"lines"),
      panel.background = element_blank(),
      panel.grid.major=element_line(linewidth = 0.05,color = "grey"),
      panel.grid.minor=element_line(linewidth = 0.05,color = "grey"),
      strip.background = element_blank()
      )+
scale_color_manual(values = color_palette)
ggsave("MEF_Last_plot.pdf", plot, width = 4, height = 3)

Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.


In [50]:
plot = plot_model(model3, type='pred', 
           terms=c("funded","ST [Advanced, Proficient,Developing,Lagging]"),
          title="",
          axis.title=c("",''),
          legend.title='Scientific Capacity')+
set_theme(axis.textsize=1.1)+
theme(axis.text.x = element_text(angle=0),
      panel.spacing=unit(3.5,"lines"),
      panel.background = element_blank(),
      panel.grid.major=element_line(linewidth = 0.05,color = "grey"),
      panel.grid.minor=element_line(linewidth = 0.05,color = "grey"),
      strip.background = element_blank()
      )+
scale_color_manual(values = color_palette)
ggsave("MEF_First_plot.pdf", plot, width = 4, height = 3)

Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.
